In [45]:
import  pandas as pd
from sklearn.preprocessing import LabelEncoder , MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier,StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score,f1_score,recall_score,precision_score

In [46]:
data = pd.read_csv("heart.csv")

In [47]:
df = data.copy()

In [48]:
df.isnull().sum()

Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
ExerciseAngina    0
Oldpeak           0
ST_Slope          0
HeartDisease      0
dtype: int64

In [49]:
data.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [50]:
print(df["ChestPainType"].unique())
print(df["RestingECG"].unique())
print(df["ST_Slope"].unique())

['ATA' 'NAP' 'ASY' 'TA']
['Normal' 'ST' 'LVH']
['Up' 'Flat' 'Down']


In [51]:
nominal_cols = ["Sex", "ExerciseAngina", "ChestPainType", "RestingECG", "ST_Slope"]
df = pd.get_dummies(df, columns=nominal_cols, dtype=int)

In [52]:
df.head()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease,Sex_F,Sex_M,ExerciseAngina_N,...,ChestPainType_ASY,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_LVH,RestingECG_Normal,RestingECG_ST,ST_Slope_Down,ST_Slope_Flat,ST_Slope_Up
0,40,140,289,0,172,0.0,0,0,1,1,...,0,1,0,0,0,1,0,0,0,1
1,49,160,180,0,156,1.0,1,1,0,1,...,0,0,1,0,0,1,0,0,1,0
2,37,130,283,0,98,0.0,0,0,1,1,...,0,1,0,0,0,0,1,0,0,1
3,48,138,214,0,108,1.5,1,1,0,0,...,1,0,0,0,0,1,0,0,1,0
4,54,150,195,0,122,0.0,0,0,1,1,...,0,0,1,0,0,1,0,0,0,1


In [53]:
y = df["HeartDisease"]
X = df.drop("HeartDisease", axis=1)
X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [54]:
X_train , X_test , y_train , y_test = [ i.reset_index(drop=True) for i in [X_train , X_test , y_train , y_test]]

In [55]:
X_train.head()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,Sex_F,Sex_M,ExerciseAngina_N,ExerciseAngina_Y,ChestPainType_ASY,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_LVH,RestingECG_Normal,RestingECG_ST,ST_Slope_Down,ST_Slope_Flat,ST_Slope_Up
0,42,120,240,1,194,0.8,0,1,1,0,0,0,1,0,0,1,0,1,0,0
1,36,130,209,0,178,0.0,0,1,1,0,0,0,1,0,0,1,0,0,0,1
2,56,150,213,1,125,1.0,0,1,0,1,1,0,0,0,0,1,0,0,1,0
3,37,130,211,0,142,0.0,1,0,1,0,0,0,1,0,0,1,0,0,0,1
4,51,120,0,1,104,0.0,0,1,1,0,1,0,0,0,0,1,0,0,1,0


In [56]:
minmax = MinMaxScaler()
minmax_scaled = minmax.fit_transform(X_train)
minmaxscaled = minmax.transform(X_test)
X_train = pd.DataFrame(minmax_scaled,columns=X_train.columns)
X_test = pd.DataFrame(minmaxscaled,columns=X_test.columns)

In [57]:
X_train.head()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,Sex_F,Sex_M,ExerciseAngina_N,ExerciseAngina_Y,ChestPainType_ASY,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_LVH,RestingECG_Normal,RestingECG_ST,ST_Slope_Down,ST_Slope_Flat,ST_Slope_Up
0,0.270833,0.60,0.398010,1.0,0.943662,0.386364,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
1,0.145833,0.65,0.346600,0.0,0.830986,0.295455,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,0.562500,0.75,0.353234,1.0,0.457746,0.409091,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,0.166667,0.65,0.349917,0.0,0.577465,0.295455,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
4,0.458333,0.60,0.000000,1.0,0.309859,0.295455,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


In [58]:
estimators = [
    ("xgb",XGBClassifier(random_state = 42)),
    ("rf",RandomForestClassifier(n_estimators=100,random_state=42)),
    ("svc",SVC(C=1,random_state=42))
]
staking_model = StackingClassifier(
    estimators=estimators,
    final_estimator = LogisticRegression(random_state=42),
    cv=5
)

In [59]:
staking_model.fit(X_train,y_train)

StackingClassifier(cv=5,
                   estimators=[('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=None,
                                              device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric=None,
                                              feature_types=None,
                                              feature_weights=None, gamma=None,
                                              grow_policy=None,
                                              importance_type=None,
                                              interaction_...
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None,
                                              max_depth=None, max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=None, n_jobs=None,
                                              num_parallel_tree=None, ...)),
                               ('rf', RandomForestClassifier(random_state=42)),
                               ('svc', SVC(C=1, random_state=42))],
                   final_estimator=LogisticRegression(random_state=42))

In [60]:
prediction = staking_model.predict(X_test)

In [61]:
print(f"Accuracy is {accuracy_score(y_test,prediction)}")
print(f"prediction is {precision_score(y_test,prediction)}")
print(f"F1 score is {f1_score(y_test,prediction)}")
print(f"Recall is {recall_score(y_test,prediction)}")

Accuracy is 0.8804347826086957
prediction is 0.9047619047619048
F1 score is 0.8962264150943396
Recall is 0.8878504672897196
